In [6]:
import os
from os.path import split

import pandas as pd
import matplotlib.pyplot as plt
from sympy import false
from ultralytics import YOLO

CONFIG_YAML = 'configs/config.yaml' 

# model = YOLO('yolov8n.pt')
# 
# results = model.train(
#     data=CONFIG_YAML,
#     epochs=1,          # 50 epochs is a great baseline for a 3-day project timeline
#     imgsz=640,          # Native resolution of the DeepPCB line-scan dataset
#     batch=32,           # RTX 4070 easily handles batch size 32 at 640x640
#     device=0,           # Forces training on your primary CUDA GPU
#     workers=4,          # Multi-threaded CPU preprocessing
#     amp=True,           # CRITICAL: Enables Automatic Mixed Precision (FP16)
#     project='PCB_AOI',  # Root folder name for saved outputs
#     name='test8'  # Specific folder name for this run
# )

model = YOLO('yolo26n.pt')

results = model.train(
    data=CONFIG_YAML,
    epochs=1,          # 50 epochs is a great baseline for a 3-day project timeline
    imgsz=640,          # Native resolution of the DeepPCB line-scan dataset
    batch=32,           # RTX 4070 easily handles batch size 32 at 640x640
    device=0,           # Forces training on your primary CUDA GPU
    workers=4,          # Multi-threaded CPU preprocessing
    amp=True,           # CRITICAL: Enables Automatic Mixed Precision (FP16)
    project='PCB_AOI',  # Root folder name for saved outputs
    name='test26'  # Specific folder name for this run
)
# model = YOLO('yolo26n.pt')
# 
results = model.train(
    data=CONFIG_YAML,
    epochs=100,
    imgsz=640,
    batch=32,
    device=0,
    patience=10,
    amp=True,
    project='PCB_AOI',
    name='yolo26_no_augmentations_raw_es',

    # ==========================================
    # COMPLETELY DEACTIVATING ALL AUGMENTATIONS
    # ==========================================
    # 1. Geometric & Spatial Transformations (Set to 0)
    degrees=0.0,       # No random rotations
    fliplr=0.0,        # No horizontal flipping (0% chance)
    flipud=0.0,        # No vertical flipping (0% chance)
    scale=0.0,         # No random zooming
    translate=0.0,     # No random panning/shifting
    shear=0.0,         # No shearing/skewing
    perspective=0.0,   # No perspective warping

    # 2. Multi-Image & Advanced Aggregations (Set to 0 / False)
    mosaic=0.0,        # DISABLED: Do not blend 4 images into a grid collage
    mixup=0.0,         # DISABLED: Do not blend images on top of each other
    copy_paste=0.0,    # DISABLED: Do not copy objects from other frames
    erasing=0.0,       # DISABLED: Do not randomly black out patches of the board
    auto_augment=None, # DISABLED: Turn off automatic policies like RandAugment

    # 3. Color and Lighting Transformations (Set to 0)
    hsv_h=0.0,         # No hue shifting
    hsv_s=0.0,         # No saturation shifting
    hsv_v=0.0,         # No brightness/value shifting
    bgr=0.0            # No channel swapping
)


# ==========================================
# 3. EXTRACTING AND VISUALIZING METRICS
# ==========================================
print("\n--- Training Complete! Parsing Metrics ---")

# Ultralytics automatically saves a CSV of metrics in the run directory
results_dir = results.save_dir
csv_path = os.path.join(results_dir, 'results.csv')

if os.path.exists(csv_path):
    # Load metrics into a Pandas DataFrame
    df = pd.read_csv(csv_path)
    # Strip whitespace from column names to prevent indexing keys error
    df.columns = df.columns.str.strip()
    
    # Setting up a side-by-side metric plot for your presentation slides
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    
    # Graph 1: Loss Convergence Curves
    axes[0].plot(df['epoch'], df['train/box_loss'], label='Train Box Loss', color='royalblue', lw=2)
    axes[0].plot(df['epoch'], df['val/box_loss'], label='Val Box Loss', color='orange', linestyle='--', lw=2)
    axes[0].set_title('Bounding Box Regression Loss Across Epochs')
    axes[0].set_xlabel('Epochs')
    axes[0].set_ylabel('Loss Value')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Graph 2: Mean Average Precision (mAP)
    axes[1].plot(df['epoch'], df['metrics/mAP50(B)'], label='mAP @ 0.5', color='emerald' if 'emerald' in plt.cm.datad else 'green', lw=2)
    axes[1].plot(df['epoch'], df['metrics/mAP50-95(B)'], label='mAP @ 0.5:0.95', color='darkred', linestyle='-.', lw=2)
    axes[1].set_title('Mean Average Precision (mAP) Trends')
    axes[1].set_xlabel('Epochs')
    axes[1].set_ylabel('Accuracy Score (0.0 to 1.0)')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Print the absolute final model scores
    final_row = df.iloc[-1]
    print("\n==================================================")
    print("           FINAL PRESENTATION METRICS             ")
    print("==================================================")
    print(f"Final Training Epoch attained:  {int(final_row['epoch'])}")
    print(f"mAP @ 0.5 (Overall Box Accuracy): {final_row['metrics/mAP50(B)']:.4f} ({final_row['metrics/mAP50(B)']*100:.1f}%)")
    print(f"mAP @ 0.5:0.95 (Strict Localization): {final_row['metrics/mAP50-95(B)']:.4f}")
    print(f"Precision Score (True Positives): {final_row['metrics/precision(B)']:.4f}")
    print(f"Recall Score (Defect Detection Rate): {final_row['metrics/recall(B)']:.4f}")
    print("==================================================")
    
else:
    print(f"Error: Could not locate results.csv at expected path: {csv_path}")

Ultralytics 8.4.67  Python-3.12.3 torch-2.12.0+cu126 CUDA:0 (NVIDIA GeForce RTX 4070, 12282MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=configs/config.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=1, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=test26, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100,

<Figure size 1500x500 with 2 Axes>


           FINAL PRESENTATION METRICS             
Final Training Epoch attained:  1
mAP @ 0.5 (Overall Box Accuracy): 0.0344 (3.4%)
mAP @ 0.5:0.95 (Strict Localization): 0.0097
Precision Score (True Positives): 0.0146
Recall Score (Defect Detection Rate): 0.3668
